# DarkPipe v0.15 — adquisición KiDS segmentada hacia Google Drive

Este notebook descarga los cuatro insumos públicos mediante rangos de 256 MiB, guarda cada shard directamente en Drive y reanuda después de una interrupción. No crea el catálogo completo en el disco del ordenador ni produce un resultado científico.


In [ ]:
%pip install -q "git+https://github.com/FacundoFirmenich/darkpipe-realdata.git@main"


In [ ]:
from google.colab import auth, drive
drive.mount('/content/drive')
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build

credentials, _ = google.auth.default()
service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
storage_quota = service.about().get(fields='storageQuota').execute()['storageQuota']
if 'limit' not in storage_quota:
    raise RuntimeError('Drive no expuso un límite finito verificable; no se inicia la transferencia.')
drive_free_bytes = int(storage_quota['limit']) - int(storage_quota['usage'])
print({'drive_free_bytes': drive_free_bytes, 'drive_free_gib': drive_free_bytes / 1024**3})


In [ ]:
from pathlib import Path
from darkpipe.drive_sharded_acquisition import (
    DEFAULT_DRIVE_SHARD_BYTES,
    DRIVE_SHARDED_JURISDICTION,
    acquire_default_inputs_to_drive,
)

DRIVE_ROOT = Path('/content/drive/MyDrive/DarkPipe/v0.15/KiDS-shards')
ACKNOWLEDGE_UPSTREAM_TERMS = True
EXECUTE_TRANSFER = True

if not EXECUTE_TRANSFER:
    raise RuntimeError('Transferencia no autorizada en esta ejecución.')

result = acquire_default_inputs_to_drive(
    DRIVE_ROOT,
    observed_drive_free_bytes=drive_free_bytes,
    acknowledge_upstream_terms=ACKNOWLEDGE_UPSTREAM_TERMS,
    execution_jurisdiction=DRIVE_SHARDED_JURISDICTION,
    shard_bytes=DEFAULT_DRIVE_SHARD_BYTES,
)
result


## Interpretación

Éxito significa custodia byte-a-byte completa y reanudable. No significa que el RAR haya sido reconstruido ni que ningún modelo físico haya sido favorecido. Los recibos quedan en `MyDrive/DarkPipe/v0.15/KiDS-shards`.
